# Construye un sistema inteligente de OCR con Amazon Textract y Python

**AWS Community Day Ecuador 2026** · Auditorio Luis Alberto Luna, UPS Cuenca · 15:00–16:10
Ponente: Manuel Josue

Este notebook es una alternativa a los scripts de terminal (`00_check.py` … `05_bonus_bedrock.py`).
Hace exactamente lo mismo, importando el mismo paquete `textract_lab`, pero muestra las imágenes
en línea: verás el **overlay de cajas de confianza** sin descargar nada.

## Cómo usarlo

| Dónde | Qué necesitas | Modo |
|---|---|---|
| Tu laptop con Jupyter o VS Code | `pip install boto3` | offline u online |
| Google Colab | nada, la primera celda clona el repo | **solo offline** |
| CloudShell | ahí no hay Jupyter: usa los scripts `.py` | — |

**Por defecto este notebook corre en modo OFFLINE**, leyendo respuestas reales de Textract ya grabadas
(`fixtures/`). No necesita credenciales, no gasta un centavo y funciona aunque se caiga el wifi.

> **Seguridad:** si estás en Google Colab, **no pegues tus claves de AWS aquí**. Colab es un entorno de
> terceros. El modo offline te deja hacer el taller completo sin credenciales. Si quieres llamar a AWS
> de verdad, hazlo desde tu laptop con un perfil (`AWS_PROFILE`), nunca pegando claves en una celda.

## 0. Preparación

Ejecuta esta celda primero. Detecta dónde estás, clona el repo si hace falta y te dice en qué modo vas.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Modo offline por defecto: sin credenciales, sin costo. Cambia a "online" solo en tu laptop.
os.environ.setdefault("LAB_MODO", "offline")

EN_COLAB = "google.colab" in sys.modules
URL_REPO = "https://github.com/jossuema/textract-ec.git"   # <-- el ponente lo reemplaza

if EN_COLAB and not Path("textract-ec").exists():
    subprocess.run(["git", "clone", "--depth", "1", URL_REPO], check=True)

# La raíz del repo es el directorio que contiene textract_lab/
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("textract-ec").resolve()]
             if (p / "textract_lab").is_dir()), None)
assert RAIZ, "No encuentro textract_lab/. Abre el notebook desde el repo clonado."
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from textract_lab import bloques, cliente, validadores, etapas, overlay, costos
from textract_lab.queries import QUERIES_FACTURA, QUERIES_ORDEN_COMPRA

print(cliente.banner_modo(cliente.resolver_modo()))
print("raíz del repo:", RAIZ.name + "/")   # sin ruta absoluta: no filtra tu usuario
print("Python", sys.version.split()[0])

Si arriba lees `[OFFLINE · fixtures reales grabados …]`, estás listo: todo el taller funciona.

Las funciones que vas a usar son las mismas de los scripts:

- `cliente.llamar(operacion, documento, ...)` decide sola si llama a Textract o lee un fixture.
- `bloques.*` navega el grafo de la respuesta.
- `validadores.*` aplica las reglas de negocio ecuatorianas.

---
## Lab 1 · Texto y confianza

`DetectDocumentText` es la operación barata (US$ 0.0015 por página). Devuelve líneas y palabras,
cada una con una **confianza de 0 a 100**.

In [ ]:
DOC = "docs/factura_limpia.png"

resp = cliente.llamar("detect_document_text", DOC)
lineas = bloques.lineas(resp)

print(f"{len(lineas)} líneas · confianza media LINE "
      f"{bloques.confianza_media(resp, 'LINE'):.1f} · WORD {bloques.confianza_media(resp, 'WORD'):.1f}")
for texto, conf in lineas[:12]:
    marca = "verde" if conf >= 90 else ("ámbar" if conf >= 70 else "ROJO")
    print(f"  {conf:5.1f}  {marca:<6}  {texto}")

### El semáforo, en color

Los umbrales que sugiere AWS: a partir de **90** puedes tomar decisiones financieras; por debajo de 70,
descarta. Aquí va la misma lista, coloreada.

In [ ]:
from IPython.display import HTML, display

def semaforo_html(lineas, limite=20):
    filas = []
    for texto, conf in lineas[:limite]:
        color = "#1B7F4B" if conf >= 90 else ("#B8730B" if conf >= 70 else "#B3312B")
        filas.append(
            f"<tr><td style='color:{color};font-weight:700;text-align:right;padding:2px 10px'>{conf:.1f}</td>"
            f"<td style='padding:2px 10px'>{texto}</td></tr>")
    return HTML("<table style='font-family:ui-monospace,Menlo,monospace;font-size:13px'>"
                + "".join(filas) + "</table>")

display(semaforo_html(lineas))

### El overlay: dónde está cada palabra y cuánta confianza tiene

`Geometry.BoundingBox` viene **normalizado de 0 a 1**: para pasar a píxeles se multiplica por el ancho
o el alto de la imagen. Esta es la parte que en CloudShell no se puede ver.

In [ ]:
from IPython.display import Image

destino = Path("salida") / "notebook_overlay.png"
destino.parent.mkdir(exist_ok=True)
ruta = overlay.dibujar(DOC, resp, destino)

if ruta:
    display(Image(filename=str(ruta), width=760))
else:
    print("Pillow no está instalado (pip install pillow). Mira docs/salidas/ para verlo ya generado.")

### Ejercicio (2 minutos)

Compara la factura original con la **misma factura fotografiada con un celular**. ¿Cuánto baja la
confianza? ¿Qué palabras concretas se ven afectadas: las que importan (montos, identificadores) o las
que no (direcciones)?

In [ ]:
for documento in ["docs/factura_limpia.png", "docs/factura_foto.jpg"]:
    r = cliente.llamar("detect_document_text", documento)
    bajas = bloques.palabras_bajo_umbral(r, 90)
    print(f"{documento:<28} WORD {bloques.confianza_media(r, 'WORD'):5.1f} · "
          f"{len(bajas)} palabras por debajo de 90")

# TU TURNO: imprime las 5 palabras de menor confianza de la foto y decide si te importan.

---
## Lab 2 · Formularios, tablas y tu único TODO

`AnalyzeDocument` con `FORMS`, `TABLES` y `LAYOUT` cuesta US$ 0.065 por página. Devuelve el mismo texto
más la estructura: pares clave-valor, celdas de tabla y regiones de la página.

La respuesta es un **grafo**: cada bloque tiene un `Id` y una lista de `Relationships`. Los hijos no
conocen a su padre, así que se construye un índice y se navega de arriba hacia abajo.

In [ ]:
resp_f = cliente.llamar("analyze_document", DOC, feature_types=["FORMS", "TABLES", "LAYOUT"])
print(bloques.resumen_bloques(resp_f))

### Tu único TODO del taller

Escribe `mi_pares_clave_valor`. Son unas 8 líneas y esta es la receta:

1. Construye el índice con `bloques.indice(resp)`.
2. Recorre `resp["Blocks"]` y quédate con los de tipo `KEY_VALUE_SET` cuyo `EntityTypes` incluya `"KEY"`.
3. Para cada uno, su valor es `bloques.hijos(b, by_id, tipo="VALUE")[0]`.
4. El texto de cualquier bloque sale de `bloques.texto(b, by_id)`.

La celda de abajo compara tu versión con la de la librería y te dice si coincide.

In [ ]:
def mi_pares_clave_valor(resp):
    by_id = bloques.indice(resp)
    pares = {}
    # TU CÓDIGO AQUÍ (unas 6 líneas)

    return pares


# ── autoverificación ─────────────────────────────────────────────────────────
mio = mi_pares_clave_valor(resp_f)
libreria = {k: c.valor for k, c in bloques.pares_clave_valor(resp_f).items()}

if not mio:
    print("TODO pendiente: usando la versión de la librería para que puedas seguir.")
    mio = libreria
elif mio == libreria:
    print(f"✔ tu implementación coincide: {len(mio)} pares")
else:
    faltan = set(libreria) - set(mio)
    sobran = set(mio) - set(libreria)
    print(f"casi: te faltan {sorted(faltan)[:5]} · te sobran {sorted(sobran)[:5]}")

print()
for clave in ["RUC", "No.", "FECHA EMISION", "VALOR TOTAL"]:
    campo = bloques.buscar_clave(bloques.pares_clave_valor(resp_f), clave)
    print(f"  {clave:<16} → {campo.valor if campo else '(no encontrado)'}")

### Tablas: `RowIndex` y `ColumnIndex`

Cada `CELL` sabe su fila y su columna (empezando en 1). Con eso se reconstruye la tabla.

In [ ]:
tablas = bloques.tablas(resp_f)
print(f"{len(tablas)} tablas detectadas")

detalle = max(tablas, key=lambda t: len(t.filas))
display(HTML("<table style='font-size:13px;border-collapse:collapse'>" + "".join(
    "<tr>" + "".join(
        f"<{'th' if i == 0 else 'td'} style='border:1px solid #ccc;padding:3px 8px'>{c}</"
        f"{'th' if i == 0 else 'td'}>" for c in fila) + "</tr>"
    for i, fila in enumerate(detalle.filas)) + "</table>"))

### Checkboxes

En un formulario, el valor de un par puede ser un `SELECTION_ELEMENT` con `SelectionStatus`
`SELECTED` o `NOT_SELECTED`. La librería lo convierte en `[X]` o `[ ]`.

In [ ]:
resp_form = cliente.llamar("analyze_document", "docs/formulario_inscripcion.png",
                           feature_types=["FORMS", "TABLES", "LAYOUT"])
for clave, estado, conf in bloques.selecciones(resp_form):
    print(f"  {'[X]' if estado == 'SELECTED' else '[ ]'}  {clave:<32} {conf:5.1f}")

---
## Lab 3 · La capa inteligente

Aquí es donde el OCR se convierte en un **sistema**. Tres ideas:

1. **Queries**: le preguntas al documento en lenguaje natural. Cuesta US$ 0.015 por página.
   Ojo: `Query.Text` solo admite **ASCII** y está documentado **solo para inglés**.
2. **Combinar fuentes**: el mismo campo puede venir de QUERIES, de FORMS o de una fila de la tabla.
   Gana el de mayor confianza, y guardamos de dónde vino.
3. **Validar el negocio**: módulo 11 del RUC y de la clave de acceso, módulo 10 de la cédula,
   que los ítems sumen el subtotal y que subtotal + IVA cuadre con el total.

In [ ]:
for q in QUERIES_FACTURA:
    print(f"  {q['Alias']:<16} {q['Text']}")

resp_q = cliente.llamar("analyze_document", DOC,
                        feature_types=["QUERIES"], queries=QUERIES_FACTURA)

print("\nRespuestas sobre un documento en ESPAÑOL:")
for alias, campo in bloques.respuestas_queries(resp_q).items():
    print(f"  {alias:<16} {campo.valor!r:<55} conf {campo.confianza:.1f}")

Las siete respondieron, aunque Queries esté documentado solo para inglés. **Funciona, pero está fuera
del soporte oficial**: por eso no dependemos de una sola fuente.

### Quién gana cada campo

Mira `CLAVE_ACCESO`: es el campo más largo (49 dígitos) y ahí Queries baja la confianza, así que gana
FORMS. Este es el argumento real de por qué se combinan fuentes.

In [ ]:
ctx = etapas.procesar_con_contexto(DOC)
print(etapas.tabla_campos(ctx["campos_canonicos"]))

print("\nquién compitió por cada campo (gana la mayor confianza):")
for alias, campo in ctx["campos_canonicos"].items():
    print(f"  {alias:<24} gana {campo.origen}  con {campo.confianza:.1f}")

### Validar el negocio, no solo el OCR

`factura_trampa.png` tiene el OCR **perfecto** y aun así está mal: el precio total de un ítem no cuadra
y la clave de acceso tiene mal el dígito verificador. Los dos errores los puso el ponente a propósito.

In [ ]:
for documento in ["docs/factura_limpia.png", "docs/factura_trampa.png", "docs/factura_foto.jpg"]:
    r = etapas.procesar_documento(documento)
    print(f"\n{documento}  →  {r.estado}")
    for a in r.alertas:
        print(f"    - {a}")

---
## Lab 4 · Esto ya es un sistema

Un pipeline es una lista de funciones que reciben y devuelven un contexto. Cambiar una etapa (por
ejemplo, reemplazar el clasificador por un LLM) no obliga a tocar el resto.

In [ ]:
print("ETAPAS =", [f.__name__ for f in etapas.ETAPAS])

documentos = sorted(p for p in Path("docs").glob("*.png")) + sorted(Path("docs").glob("*.jpg"))
filas = [["documento", "tipo", "estado", "alertas"]]
for d in documentos:
    try:
        r = etapas.procesar_documento(str(d))
        filas.append([d.name, r.tipo_documento, r.estado, str(len(r.alertas))])
    except cliente.FixtureFaltante:
        filas.append([d.name, "—", "SIN DATOS", "—"])

display(HTML("<table style='font-size:13px;border-collapse:collapse'>" + "".join(
    "<tr>" + "".join(
        f"<{'th' if i == 0 else 'td'} style='border:1px solid #ccc;padding:3px 10px;text-align:left'>{c}</"
        f"{'th' if i == 0 else 'td'}>" for c in fila) + "</tr>"
    for i, fila in enumerate(filas)) + "</table>"))

---
## Bonus · Cuándo Textract no basta

`orden_compra_prosa.png` es una carta: no tiene etiquetas ni tablas. Mira qué contestan las Queries.

In [ ]:
resp_prosa = cliente.llamar("analyze_document", "docs/orden_compra_prosa.png",
                            feature_types=["QUERIES"], queries=QUERIES_ORDEN_COMPRA)
for alias, campo in bloques.respuestas_queries(resp_prosa).items():
    valor = campo.valor or "(sin respuesta)"
    print(f"  {alias:<16} {valor!r:<26} conf {campo.confianza:.1f}")

print("\nLo que dice la carta de verdad:")
print("  proveedor: Tecnología Andina Ejemplo S.A. · monitores: 3 · total: 1221.20")

Fíjate bien en lo que acaba de pasar, porque es la lección más importante del taller:

- `CANT_MONITORES` responde **27** con **99 % de confianza**. Pero 27 son las **pulgadas** del monitor;
  la cantidad es 3.
- `PROVEEDOR` responde **"Gerente de Compras"**, que es el cargo de quien firma la carta, no el proveedor.

Sobre prosa, Queries **no dice "no sé": responde con seguridad y se equivoca**. Una confianza alta
significa "leí bien estos caracteres", no "entendí la pregunta".

Ahí es donde entra un LLM sobre el texto linealizado. Y ahí mismo está la regla que no se negocia:
**el LLM propone, el módulo 11 dispone**. Toda salida del modelo vuelve a pasar por los validadores y
cualquier corrección se etiqueta como alerta para un revisor humano.

El paso opcional con Amazon Bedrock está en `05_bonus_bedrock.py` y la receta para habilitarlo en tu
cuenta, en `bonus/README.md`.

---
## Para llevarte a casa

- `extra/06_expense.py` — `AnalyzeExpense`, la API específica de facturas (oficialmente solo en inglés).
- `extra/07_async_s3.py` — el modo asíncrono con S3, para PDF de hasta 3.000 páginas.
- `extra/08_textractor.py` — la librería `amazon-textract-textractor`, con export a pandas.
- `pytest tests/` corre al 100 % en offline: puedes seguir trabajando sin cuenta AWS.

**Y antes de irte:** si creaste claves de acceso para el taller, bórralas ahora.